# 03 - Web search via Tavily (minimal payload)

Isolated demo of a *client-side* search tool - no calculator, no memory. Unlike Anthropic's built-in `web_search` (which attaches a large per-result verification blob we can't shrink), Tavily returns plain JSON that we control directly: `max_results=3`, `chunks_per_source=1` (<=500 chars/snippet), and a short synthesized `answer` instead of raw pages.

You'll need a free Tavily API key from https://tavily.com (their free tier includes search credits).

## 1. Install dependencies

In [ ]:
%pip install -q anthropic requests

## 2. Set your API keys

Both use `getpass` so neither key is saved in plain text in the notebook file.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter your TAVILY_API_KEY: ")

## 3. Tools, the agent loop, and the `Agent` class

Same cost cap and turn-collapsing core as the other notebooks. The collapsing step matters less here than it did for Anthropic's built-in tool (Tavily's payload is already small), but it's kept for consistency and because it still helps on very long conversations.

In [ ]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass
import requests

TAVILY_MAX_RESULTS = 3  # keep this small - it directly controls tokens spent

SYSTEM_PROMPT = (
    "You are a helpful assistant with a web_search tool. Use it for "
    "current events or anything you're not sure about; otherwise reply "
    "directly."
)

TOOLS = [
    {
        "name": "web_search",
        "description": "Search the web for current information. Returns a short synthesized answer plus a few source snippets.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query.",
                },
            },
            "required": ["query"],
        },
    },
]


def web_search(query: str) -> str:
    """Client-side search via Tavily's REST API - unlike Anthropic's
    built-in web_search tool, we get plain JSON back and control exactly
    how much text comes through (no per-result verification blob)."""
    api_key = os.environ["TAVILY_API_KEY"]
    try:
        resp = requests.post(
            "https://api.tavily.com/search",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "query": query,
                "max_results": TAVILY_MAX_RESULTS,
                "chunks_per_source": 1,  # <=500 chars per source snippet
                "include_answer": "basic",  # short synthesized answer
            },
            timeout=15,
        )
        resp.raise_for_status()
    except requests.RequestException as exc:
        return f"Error: web search failed ({exc})"

    data = resp.json()
    lines = []
    answer = data.get("answer")
    if answer:
        lines.append("Answer: " + answer)
    for r in data.get("results", []):
        lines.append("- " + r["title"] + " (" + r["url"] + "): " + r["content"])
    return "\n".join(lines) if lines else "No results found."


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "web_search":
        return web_search(tool_input["query"])
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or anthropic.Anthropic()
        self.messages: list[dict] = []
        self.total_cost_usd = 0.0

    def send(self, user_input: str) -> str:
        turn_start = len(self.messages)
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                # A server-side tool hit its per-turn iteration limit;
                # resend as-is to let Claude continue.
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        reply = "".join(
            block.text for block in response.content if block.type == "text"
        )

        # Collapse this turn's tool exchange down to plain {user, assistant}
        # text. Anything left in self.messages gets resent on every future
        # turn (the API is stateless) - the visible answer is all future
        # turns need.
        self.messages[turn_start:] = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": reply},
        ]
        return reply


## 4. Create the agent

In [ ]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

## 5. Try it

In [ ]:
reply = agent.send("Briefly, what is the latest news about SpaceX?")
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

## 6. Optional: interactive chat loop

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")
